In [7]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
os.environ.get("OPENAI_API_KEY") is not None

True

In [50]:
df = pd.read_csv("netflix_titles_1000.csv")

In [51]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [43]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')

In [11]:
client = chromadb.PersistentClient(
    path=r"C:\Users\jorgeeyy\Documents\projects\ai-engineer-devs-track\chroma_db"
)

In [12]:
client

In [13]:
embedding_function = OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small",
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [44]:
collection = client.get_or_create_collection(
    name="netflix_titles_1000",
    embedding_function=embedding_function,
)

In [45]:
collection

Collection(name=netflix_titles_1000)

In [46]:
collection.count()

0

In [52]:
row = df.iloc[4]

document = f"""
Title: {row['title']}
Type: {row['type']}
Director: {row['director']}
Cast: {row['cast']}
Country: {row['country']}
Release Year: {row['release_year']}
Rating: {row['rating']}
Duration: {row['duration']}
Genres: {row['listed_in']}
Description: {row['description']}
"""

In [53]:
print(document)


Title: Kota Factory
Type: TV Show
Director: nan
Cast: Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar
Country: India
Release Year: 2021
Rating: TV-MA
Duration: 2 Seasons
Genres: International TV Shows, Romantic TV Shows, TV Comedies
Description: In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life.



In [21]:
collection.add(
    ids=["netflix-0"],
    documents=[document],
)

In [22]:
collection.count()

1

In [26]:
results = collection.get()
results['documents']

['\nTitle: Dick Johnson Is Dead\nType: Movie\nDirector: Kirsten Johnson\nCast: nan\nCountry: United States\nRelease Year: 2020\nRating: PG-13\nDuration: 90 min\nGenres: Documentaries\nDescription: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.\n']

In [27]:
results = collection.query(
    query_texts=["a documentary about someone's father"],
    n_results=1,
)

In [28]:
results['documents']

[['\nTitle: Dick Johnson Is Dead\nType: Movie\nDirector: Kirsten Johnson\nCast: nan\nCountry: United States\nRelease Year: 2020\nRating: PG-13\nDuration: 90 min\nGenres: Documentaries\nDescription: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.\n']]

In [54]:
documents = []

for index, row in df.iterrows():
    document = f"""
Title: {row['title']}
Type: {row['type']}
Director: {row['director']}
Cast: {row['cast']}
Country: {row['country']}
Release Year: {row['release_year']}
Rating: {row['rating']}
Duration: {row['duration']}
Genres: {row['listed_in']}
Description: {row['description']}
"""
    
    documents.append(document)

In [55]:
len(documents)

1000

In [56]:
ids = [f"netflix-{i}" for i in range(len(documents))]

In [57]:
ids[:10]

['netflix-0',
 'netflix-1',
 'netflix-2',
 'netflix-3',
 'netflix-4',
 'netflix-5',
 'netflix-6',
 'netflix-7',
 'netflix-8',
 'netflix-9']

In [58]:
collection.upsert(
    ids=ids,
    documents=documents
)
collection.count()

1000

In [59]:
results = collection.query(
    query_texts=["movie about a deadly home invasion"],
    n_results=3,
)

In [35]:
print(results['documents'])

[['\nTitle: Ganglands\nType: TV Show\nDirector: Julien Leclercq\nCast: Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera\nCountry: nan\nRelease Year: 2021\nRating: TV-MA\nDuration: 1 Season\nGenres: Crime TV Shows, International TV Shows, TV Action & Adventure\nDescription: To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war.\n']]


In [61]:
for document in results["documents"][0]:
    print(document)
    print("=" * 80)


Title: Intrusion
Type: Movie
Director: Adam Salky
Cast: Freida Pinto, Logan Marshall-Green, Robert John Burke, Megan Elisabeth Kelly, Sarah Minnich, Hayes Hargrove, Mark Sivertsen, Brandon Fierro, Antonio Valles, Clint Obenchain
Country: nan
Release Year: 2021
Rating: TV-14
Duration: 94 min
Genres: Thrillers
Description: After a deadly home invasion at a couple’s new dream house, the traumatized wife searches for answers — and learns the real danger is just beginning.


Title: The Strangers: Prey at Night
Type: Movie
Director: Johannes Roberts
Cast: Bailee Madison, Martin Henderson, Emma Bellomy, Lea Enslin, Damian Maffei, Christina Hendricks, Lewis Pullman
Country: United States, United Kingdom
Release Year: 2018
Rating: R
Duration: 85 min
Genres: Horror Movies
Description: Home-invasion horror comes to the trailer park when a family arrives at a mobile home community to find nobody there but a trio of masked murderers.


Title: Deadly Switch
Type: Movie
Director: Svetlana Cvetko
Cas